# Classification Trees, Random Forest and Gradient Boosting – Concepts

This notebook summarises the concepts and adds explanations and small examples. All examples use small hand-made numbers or simulated data – no data set is loaded.

**Contents**
1. Anatomy of a classification tree
2. Impurity: Gini and entropy
3. Worked example of a split
4. Growing and predicting
5. Random forest and gradient boosting for classification
6. Putting it together in scikit-learn

## Libraries and settings

In [ ]:
# Libraries
import os
import platform
import warnings
from datetime import datetime
from platform import python_version

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, roc_auc_score

# Ignore warnings
warnings.filterwarnings('ignore')

## 1. Anatomy of a classification tree

An example tree (Titanic):

```
male?
├── yes: age ≤ 9.5?
│   ├── yes: survived
│   └── no:  not survived
└── no:  3rd class?
    ├── no:  survived
    └── yes: fare ≤ 23?
        ├── yes: survived
        └── no:  not survived
```

- **Root node:** the first question (here: *male?*), containing all observations.
- **Internal (child) nodes:** further questions.
- **Terminal nodes (leaves):** the predictions.
- Each internal node asks a yes/no question; the branches are labelled yes/no.
- **Depth** $d$ = longest root→leaf path (here 3). **Size** = number of nodes (here 9).
- A binary tree of depth $d$ has at most $2^{d+1} - 1$ nodes (for $d = 3$: 15) and at most $2^d$ leaves (8).

## 2. Impurity: Gini and entropy

For a node with class shares $\hat p_k$ ($k = 1, \dots, K$):

$$\text{Gini: } G = 1 - \sum_{k=1}^{K}\hat p_k^2 \qquad\qquad \text{Entropy: } H = -\sum_{k=1}^{K}\hat p_k \log_2 \hat p_k$$

$G = 0$: pure node (only one class); $G = 0.5$: maximally mixed (two classes, 50 : 50). Entropy is an alternative
measure with almost the same shape (maximum 1 for two classes).

Example with 10 observations (yes : no):

In [ ]:
# Gini impurity and entropy for two classes
def gini(p):
    """Gini impurity of a node with share p of class 'yes'."""
    return 1 - (p**2 + (1 - p) ** 2)


def entropy(p):
    """Entropy of a node with share p of class 'yes'."""
    if 0 < p < 1:
        return -(p * np.log2(p) + (1 - p) * np.log2(1 - p))
    return 0.0


print(
    pd.DataFrame(
        {
            'yes : no': [f'{k} : {10 - k}' for k in range(0, 11)],
            'Gini': [round(gini(k / 10), 2) for k in range(0, 11)],
            'entropy': [round(entropy(k / 10), 2) for k in range(0, 11)],
        }
    ).to_string(index=False)
)

p = np.linspace(0.001, 0.999, 300)
plt.figure(figsize=(6, 3.5))
plt.plot(p, gini(p), label='Gini')
plt.plot(p, [entropy(v) / 2 for v in p], label='entropy / 2 (scaled)')
plt.xlabel('share of class "yes" in the node')
plt.ylabel('impurity')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 3. Worked example of a split

$$\Delta G = G_{\text{parent}} - \Big(\frac{n_L}{n}\,G_L + \frac{n_R}{n}\,G_R\Big)$$

In words: impurity before the split minus the size-weighted impurity of the two children. Bigger $\Delta G$ = better
split.

Parent node: 100 passengers, 40 survived / 60 not → $G = 1 - (0.4^2 + 0.6^2) = 0.48$.

| candidate split | left child (yes) | $G_L$ | right child (no) | $G_R$ |
|---|---|---|---|---|
| male? | 60: 12 surv. / 48 not | 0.320 | 40: 28 / 12 | 0.420 |
| 1st class? | 25: 15 / 10 | 0.480 | 75: 25 / 50 | 0.444 |

In [ ]:
# Worked example: weighted Gini of two candidate splits
def gini_counts(a, b):
    """Gini impurity of a node with class counts a and b."""
    n = a + b
    return 1 - ((a / n) ** 2 + (b / n) ** 2)


g_parent = gini_counts(40, 60)
splits = [
    ('male?', (12, 48), (28, 12)),
    ('1st class?', (15, 10), (25, 50)),
]
for name, left, right in splits:
    n_l, n_r = sum(left), sum(right)
    g_l, g_r = gini_counts(*left), gini_counts(*right)
    g_w = n_l / 100 * g_l + n_r / 100 * g_r
    print(
        f'{name:11s}: G_L = {g_l:.3f}, G_R = {g_r:.3f}, '
        f'weighted G = {g_w:.3f}, Delta G = {g_parent - g_w:.3f}'
    )

**Decision:** *male?* reduces the impurity most ($\Delta G = 0.120$ vs. $0.027$) → it becomes the root split. The search
is repeated recursively in each child node.

## 4. Growing and predicting

1. For every feature and every threshold, compute the impurity decrease of the split.
2. Pick the (feature, threshold) with the largest decrease and split the node.
3. Repeat recursively in each child node.
4. Stop when a rule applies (max. depth, min. samples, pure node).
5. Optionally prune back the large tree (cost-complexity pruning, `ccp_alpha` – as for regression trees).

**Prediction in a leaf $m$:** the share of each class in the leaf, $\hat p_{mk} = \frac{1}{n_m}\sum_{x_i \in R_m} 1(y_i = k)$,
→ predict the most frequent class. Example: a leaf with 145 survived and 125 not survived →
$\hat p(\text{survived}) = 145/270 \approx 0.54$ → "survived".

**Key idea:** trees output **probabilities**, not only labels – the basis for thresholds and ROC curves. A tree grown
until all leaves are pure only outputs probabilities 0 and 1.

In [ ]:
# Class probability in a leaf
print('Leaf 145 : 125 -> P(survived) =', round(145 / 270, 3))

## 5. Random forest and gradient boosting for classification

**Random forest classifier**
- many deep classification trees (bootstrap + random features, default $m = \sqrt p$),
- each tree gives class probabilities (the shares in its leaf); the forest **averages** them → class with the highest
  mean probability (`RandomForestClassifier`).

**Gradient boosting classifier**
- many small trees, built one after the other,
- each tree fits the pseudo-residuals (negative gradient) of the **log-loss** (`GradientBoostingClassifier`).

Example: 3 trees give P(survived) = 0.8, 0.6, 0.3 → mean 0.57 → predict "survived" (≥ 0.5).

**Key idea:** the same ensemble principles as for regression – only the trees and the aggregation (class probabilities
instead of numbers) change.

In [ ]:
# Random forest: average the probabilities of the trees
p_trees = np.array([0.8, 0.6, 0.3])
print(
    'mean probability =',
    round(p_trees.mean(), 2),
    '-> survived' if p_trees.mean() >= 0.5 else '-> not survived',
)

## 6. Putting it together in scikit-learn

The following code combines all steps:

| code | concept |
|---|---|
| `train_test_split(..., test_size=0.2, stratify=y)` | hold-out split (Part 2); `stratify` keeps the class shares |
| `GridSearchCV(..., cv=5, scoring='roc_auc')` | k-fold cross-validation for the hyperparameters (Part 2) |
| `RandomForestClassifier` | classification ensemble (Part 4) |
| `min_samples_leaf`, `max_features` | tree complexity, random features (Part 3) |
| `predict_proba(...)[:, 1] >= 0.5` | threshold (Part 4) |
| `roc_auc_score`, `confusion_matrix` | ROC/AUC and confusion matrix (Part 4) |

Below, this code is run on a simulated data set (`make_classification`).

In [ ]:
# Grid search for a random forest on simulated data
X, y = make_classification(
    n_samples=800, n_features=8, n_informative=4, random_state=0
)

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    {"max_features": ["sqrt", 0.5], "min_samples_leaf": [1, 5, 20]},
    cv=5,
    scoring="roc_auc",
)
grid.fit(X_tr, y_tr)
p = grid.predict_proba(X_te)[:, 1]
print(grid.best_params_)
print(roc_auc_score(y_te, p))
print(confusion_matrix(y_te, p >= 0.5))

**Key messages**
- A classification tree chooses the split with the largest decrease of the Gini impurity (or entropy).
- Leaves output class shares – trees provide probabilities.
- Random forests average the probabilities of many de-correlated trees; boosting adds small trees that correct the errors
  of the previous ones.

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
# Show operating system, date/time and Python version
print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')